## m steering

In [1]:
import glob
import numpy as np
from scipy.linalg import fractional_matrix_power
import time

def read_vectors_glob(pattern: str) -> np.array:
    results = []
    for path in glob.iglob(pattern):
        results.extend(np.load(path, allow_pickle=True))
    return results

In [2]:
neg_vectors = read_vectors_glob('../hidden_states/pos_vectors_horse_*.npy')
# pos_vectors = read_vectors_glob('../hidden_states/pos_vectors_motorcycle_*.npy')

In [ ]:
len(neg_vectors)

In [21]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            steering_vector = np.mean(pos, axis=0) - np.mean(neg, axis=0)
            print(np.linalg.norm(steering_vector))
            steering_vector /= np.linalg.norm(steering_vector)
            result[denoising_step][block].append(steering_vector.astype(np.float32))

Processing step=0, block=down, layer=0
0.8463489921532559
Processing step=0, block=down, layer=1
0.1621948743696451
Processing step=0, block=down, layer=2
0.10432178063395144
Processing step=0, block=down, layer=3
0.2047429447471583
Processing step=0, block=down, layer=4
2.4517617720495575
Processing step=0, block=down, layer=5
5.789367879876272
Processing step=0, block=down, layer=6
2.901707969644542
Processing step=0, block=down, layer=7
3.645370357342511
Processing step=0, block=down, layer=8
1.4228674703876638
Processing step=0, block=down, layer=9
0.45924506090227246
Processing step=0, block=down, layer=10
0.2746305385361668
Processing step=0, block=down, layer=11
0.4036199030099248
Processing step=0, block=down, layer=12
0.29216463475724525
Processing step=0, block=down, layer=13
0.27570505083063707
Processing step=0, block=down, layer=14
5.171917314277964
Processing step=0, block=down, layer=15
12.452278173198932
Processing step=0, block=down, layer=16
10.366980634802694
Process

In [22]:
import pickle

with open('../steering_vectors/horse_to_motorcycle_laion_steering_vectors_normed.pickle', 'wb') as fout:
    pickle.dump(result, fout)

# cov

In [2]:
vectors = read_vectors_glob('../hidden_states/pos_vectors_horse_*.npy')

In [4]:
result = {}
for denoising_step in vectors[0].keys():
    result[denoising_step] = {}
    for block in vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            vec = np.stack([vector[denoising_step][block][layer] for vector in vectors]).astype(np.float64)
            vec /= np.linalg.norm(vec, axis=1, keepdims=True)

            mu = np.mean(vec, axis=0)
            sigma = np.dot(vec.T, vec) / (vec.shape[0] - 1)
            sigma -= np.outer(mu, mu)
            print(f'Took: {time.time() - start:.2f} s')
            
            result[denoising_step][block].append(sigma.astype(np.float32))

Processing step=0, block=down, layer=0
Took: 0.17 s
Processing step=0, block=down, layer=1
Took: 0.11 s
Processing step=0, block=down, layer=2
Took: 0.11 s
Processing step=0, block=down, layer=3
Took: 0.11 s
Processing step=0, block=down, layer=4
Took: 0.30 s
Processing step=0, block=down, layer=5
Took: 0.25 s
Processing step=0, block=down, layer=6
Took: 0.23 s
Processing step=0, block=down, layer=7
Took: 0.23 s
Processing step=0, block=down, layer=8
Took: 0.23 s
Processing step=0, block=down, layer=9
Took: 0.22 s
Processing step=0, block=down, layer=10
Took: 0.22 s
Processing step=0, block=down, layer=11
Took: 0.22 s
Processing step=0, block=down, layer=12
Took: 0.22 s
Processing step=0, block=down, layer=13
Took: 0.23 s
Processing step=0, block=down, layer=14
Took: 0.23 s
Processing step=0, block=down, layer=15
Took: 0.23 s
Processing step=0, block=down, layer=16
Took: 0.22 s
Processing step=0, block=down, layer=17
Took: 0.22 s
Processing step=0, block=down, layer=18
Took: 0.23 s
Pro

In [5]:
import pickle

with open('../horse_cov.pickle', 'wb') as fout:
    pickle.dump(result, fout)

## mm steering

In [23]:
def fractional_matrix_power_cov(A: np.ndarray, p: float, eps=1e-10):
    evals, evecs = np.linalg.eigh(A)
    evals = np.maximum(evals, 0)
    mask = (evals >= eps)
    evals = evals[mask]
    evecs = evecs[:, mask]
    return evecs @ np.diag(evals ** p) @ evecs.T

In [24]:
# pos_vectors, neg_vectors = neg_vectors, pos_vectors

In [25]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            # pos /= np.linalg.norm(pos, axis=1, keepdims=True)

            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            # neg /= np.linalg.norm(neg, axis=1, keepdims=True)

            mu_pos = np.mean(pos, axis=0)
            sigma_pos = np.dot(pos.T, pos) / (pos.shape[0] - 1)
            sigma_pos -= np.outer(mu_pos, mu_pos)

            mu_neg = np.mean(neg, axis=0)
            sigma_neg = np.dot(neg.T, neg) / (neg.shape[0] - 1)
            sigma_neg -= np.outer(mu_neg, mu_neg)

            
            sigma_neg_half = fractional_matrix_power_cov(sigma_neg, 0.5)
            sigma_neg_minus_half = fractional_matrix_power_cov(sigma_neg, -0.5)
            W = fractional_matrix_power_cov(sigma_neg_half @ sigma_pos @ sigma_neg_half, 0.5)
            W = sigma_neg_minus_half @ W @ sigma_neg_minus_half

            b = - W @ mu_neg + mu_pos

            print(f'Took: {time.time() - start:.2f} s')

            if W.dtype == np.complex128:
                print(f'Got unexpected complex values for step={denoising_step}, block={block}, layer={layer}, truncating...')
                W = np.real(W)
                b = np.real(b)
            
            result[denoising_step][block].append((W.astype(np.float32), b.astype(np.float32)))

Processing step=0, block=down, layer=0
Took: 0.33 s
Processing step=0, block=down, layer=1
Took: 0.28 s
Processing step=0, block=down, layer=2
Took: 0.28 s
Processing step=0, block=down, layer=3
Took: 0.28 s
Processing step=0, block=down, layer=4
Took: 1.04 s
Processing step=0, block=down, layer=5
Took: 0.94 s
Processing step=0, block=down, layer=6
Took: 0.93 s
Processing step=0, block=down, layer=7
Took: 0.93 s
Processing step=0, block=down, layer=8
Took: 0.94 s
Processing step=0, block=down, layer=9
Took: 0.94 s
Processing step=0, block=down, layer=10
Took: 0.96 s
Processing step=0, block=down, layer=11
Took: 0.94 s
Processing step=0, block=down, layer=12
Took: 0.94 s
Processing step=0, block=down, layer=13
Took: 0.96 s
Processing step=0, block=down, layer=14
Took: 0.93 s
Processing step=0, block=down, layer=15
Took: 0.93 s
Processing step=0, block=down, layer=16
Took: 0.93 s
Processing step=0, block=down, layer=17
Took: 0.94 s
Processing step=0, block=down, layer=18
Took: 0.94 s
Pro

In [26]:
result[0]['mid'][5][0]

array([[ 0.7864588 ,  0.0101397 ,  0.01199468, ..., -0.00631039,
        -0.02602464,  0.02117394],
       [ 0.0101397 ,  0.792389  ,  0.00188746, ...,  0.01337067,
        -0.02295238,  0.01678473],
       [ 0.01199468,  0.00188746,  0.7837137 , ...,  0.00908979,
         0.01465086, -0.00564717],
       ...,
       [-0.00631039,  0.01337067,  0.00908979, ...,  0.7982199 ,
        -0.00380174, -0.01305505],
       [-0.02602464, -0.02295238,  0.01465086, ..., -0.00380174,
         0.7814113 , -0.00236407],
       [ 0.02117394,  0.01678473, -0.00564717, ..., -0.01305505,
        -0.00236407,  0.7481372 ]], shape=(1280, 1280), dtype=float32)

In [304]:
for layer in ['down', 'mid', 'up']:
    for idx in range(len(result2[0][layer])):
        W, b = result[0][layer][idx]
        print(f'Layer {layer:4}, {idx:2}: |W|_2 = {np.linalg.norm(W, ord=2):.3f}, |b|_2 = {np.linalg.norm(b):.3f}')
        # print(np.linalg.svdvals(W)[:10])

Layer down,  0: |W|_2 = 3.415, |b|_2 = 0.672
Layer down,  1: |W|_2 = 3.089, |b|_2 = 0.163
Layer down,  2: |W|_2 = 2.858, |b|_2 = 0.129
Layer down,  3: |W|_2 = 3.135, |b|_2 = 0.265
Layer down,  4: |W|_2 = 5.893, |b|_2 = 2.081
Layer down,  5: |W|_2 = 7.296, |b|_2 = 5.868
Layer down,  6: |W|_2 = 7.292, |b|_2 = 3.300
Layer down,  7: |W|_2 = 6.730, |b|_2 = 2.701
Layer down,  8: |W|_2 = 6.904, |b|_2 = 1.368
Layer down,  9: |W|_2 = 5.509, |b|_2 = 0.584
Layer down, 10: |W|_2 = 4.878, |b|_2 = 0.312
Layer down, 11: |W|_2 = 4.699, |b|_2 = 0.300
Layer down, 12: |W|_2 = 4.618, |b|_2 = 0.321
Layer down, 13: |W|_2 = 4.609, |b|_2 = 0.867
Layer down, 14: |W|_2 = 6.601, |b|_2 = 5.933
Layer down, 15: |W|_2 = 7.276, |b|_2 = 13.119
Layer down, 16: |W|_2 = 7.483, |b|_2 = 10.802
Layer down, 17: |W|_2 = 7.496, |b|_2 = 5.679
Layer down, 18: |W|_2 = 7.978, |b|_2 = 4.107
Layer down, 19: |W|_2 = 7.090, |b|_2 = 3.995
Layer down, 20: |W|_2 = 7.035, |b|_2 = 3.086
Layer down, 21: |W|_2 = 7.221, |b|_2 = 3.092
Layer do

In [27]:
import pickle

with open('../steering_vectors/horse_to_motorcycle_laion_mm_steering_vectors_unnormed.pickle', 'wb') as fout:
    pickle.dump(result, fout)

# TODO:
- [x] Forward CASteer no norm
- [x] Inverse CASteer (normalized) with beta = 1
- [ ] CUDA libsvd NANs
- [ ] Horse to Motorcycle interpolation (forward CASteer with normalized, mmsteer unnormalized)

In [ ]:
|a| |b| cos (a, b)

In [28]:
pos_vectors, neg_vectors = neg_vectors, pos_vectors

In [29]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            # pos /= np.linalg.norm(pos, axis=1, keepdims=True)

            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            # neg /= np.linalg.norm(neg, axis=1, keepdims=True)

            mu_pos = np.mean(pos, axis=0)
            sigma_pos = np.dot(pos.T, pos) / (pos.shape[0] - 1)
            sigma_pos -= np.outer(mu_pos, mu_pos)

            mu_neg = np.mean(neg, axis=0)
            sigma_neg = np.dot(neg.T, neg) / (neg.shape[0] - 1)
            sigma_neg -= np.outer(mu_neg, mu_neg)

            
            sigma_neg_half = fractional_matrix_power_cov(sigma_neg, 0.5)
            sigma_neg_minus_half = fractional_matrix_power_cov(sigma_neg, -0.5)
            W = fractional_matrix_power_cov(sigma_neg_half @ sigma_pos @ sigma_neg_half, 0.5)
            W = sigma_neg_minus_half @ W @ sigma_neg_minus_half

            b = - W @ mu_neg + mu_pos

            print(f'Took: {time.time() - start:.2f} s')

            if W.dtype == np.complex128:
                print(f'Got unexpected complex values for step={denoising_step}, block={block}, layer={layer}, truncating...')
                W = np.real(W)
                b = np.real(b)
            
            result[denoising_step][block].append((W.astype(np.float32), b.astype(np.float32)))

Processing step=0, block=down, layer=0
Took: 0.40 s
Processing step=0, block=down, layer=1
Took: 0.33 s
Processing step=0, block=down, layer=2
Took: 0.29 s
Processing step=0, block=down, layer=3
Took: 0.28 s
Processing step=0, block=down, layer=4
Took: 0.97 s
Processing step=0, block=down, layer=5
Took: 1.05 s
Processing step=0, block=down, layer=6
Took: 0.95 s
Processing step=0, block=down, layer=7
Took: 0.97 s
Processing step=0, block=down, layer=8
Took: 1.01 s
Processing step=0, block=down, layer=9
Took: 0.97 s
Processing step=0, block=down, layer=10
Took: 0.95 s
Processing step=0, block=down, layer=11
Took: 0.95 s
Processing step=0, block=down, layer=12
Took: 0.96 s
Processing step=0, block=down, layer=13
Took: 0.95 s
Processing step=0, block=down, layer=14
Took: 0.95 s
Processing step=0, block=down, layer=15
Took: 0.96 s
Processing step=0, block=down, layer=16
Took: 0.94 s
Processing step=0, block=down, layer=17
Took: 0.94 s
Processing step=0, block=down, layer=18
Took: 0.95 s
Pro

In [30]:
import pickle

with open('../steering_vectors/motorcycle_to_horse_laion_mm_steering_vectors_unnormed.pickle', 'wb') as fout:
    pickle.dump(result, fout)

## Re-LAION

In [1]:
from datasets import load_dataset

ds = load_dataset(
    "laion/relaion2B-en-research",
    cache_dir='../cache',
    data_files=[
        f'part-{i:05}-b31ba513-fc6b-4450-9ba4-a1bba183f408-c000.snappy.parquet'
        for i in range(5)
    ],
    columns=['caption'],  # Specify only the caption column to load
)

/Users/astepanov/repos/mmsteer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
caption = ds['train']['caption']

In [7]:
import re

def generate_concept(dataset: list[str], concept: str) -> tuple[list[str], list[str]]:
    pattern = re.compile(f'(^|[\\s.,-:;]){concept}($|[\\s.,-:;])', flags=re.IGNORECASE)
    pos, neg = [], []
    for text in dataset:
        if text is None:
            continue
        if pattern.search(text) is not None:
            pos.append(text)
        else:
            neg.append(text)
    return pos, neg

In [8]:
pos_sent, neg_sent = generate_concept(caption, 'snoopy')

In [12]:
len(pos_sent)

9038

In [13]:
len(neg_sent)

84470715

In [14]:
with open('../concept_prompts/snoopy_pos_sentence.txt', 'w') as fout:
    for s in pos_sent[:9000]:
        print(s, file=fout)

In [15]:
with open('../concept_prompts/snoopy_neg_sentence.txt', 'w') as fout:
    for s in neg_sent[:9000]:
        print(s, file=fout)

In [57]:
with open('CASteer/pos_sentence.txt', 'r') as fin:
    prompts_pos = list(map(str.strip, fin.readlines()))

In [93]:
1

1

### Prompts for metrics

In [94]:
imagenet_templates = [
    'a bad photo of a {}.',
    'a photo of many {}.',
    'a sculpture of a {}.',
    'a photo of the hard to see {}.',
    'a low resolution photo of the {}.',
    'a rendering of a {}.',
    'graffiti of a {}.',
    'a bad photo of the {}.',
    'a cropped photo of the {}.',
    'a tattoo of a {}.',
    'the embroidered {}.',
    'a photo of a hard to see {}.',
    'a bright photo of a {}.',
    'a photo of a clean {}.',
    'a photo of a dirty {}.',
    'a dark photo of the {}.',
    'a drawing of a {}.',
    'a photo of my {}.',
    'the plastic {}.',
    'a photo of the cool {}.',
    'a close-up photo of a {}.',
    'a black and white photo of the {}.',
    'a painting of the {}.',
    'a painting of a {}.',
    'a pixelated photo of the {}.',
    'a sculpture of the {}.',
    'a bright photo of the {}.',
    'a cropped photo of a {}.',
    'a plastic {}.',
    'a photo of the dirty {}.',
    'a jpeg corrupted photo of a {}.',
    'a blurry photo of the {}.',
    'a photo of the {}.',
    'a good photo of the {}.',
    'a rendering of the {}.',
    'a {} in a video game.',
    'a photo of one {}.',
    'a doodle of a {}.',
    'a close-up photo of the {}.',
    'a photo of a {}.',
    'the origami {}.',
    'the {} in a video game.',
    'a sketch of a {}.',
    'a doodle of the {}.',
    'a origami {}.',
    'a low resolution photo of a {}.',
    'the toy {}.',
    'a rendition of the {}.',
    'a photo of the clean {}.',
    'a photo of a large {}.',
    'a rendition of a {}.',
    'a photo of a nice {}.',
    'a photo of a weird {}.',
    'a blurry photo of a {}.',
    'a cartoon {}.',
    'art of a {}.',
    'a sketch of the {}.',
    'a embroidered {}.',
    'a pixelated photo of a {}.',
    'itap of the {}.',
    'a jpeg corrupted photo of the {}.',
    'a good photo of a {}.',
    'a plushie {}.',
    'a photo of the nice {}.',
    'a photo of the small {}.',
    'a photo of the weird {}.',
    'the cartoon {}.',
    'art of the {}.',
    'a drawing of the {}.',
    'a photo of the large {}.',
    'a black and white photo of a {}.',
    'the plushie {}.',
    'a dark photo of a {}.',
    'itap of a {}.',
    'graffiti of the {}.',
    'a toy {}.',
    'itap of my {}.',
    'a photo of a cool {}.',
    'a photo of a small {}.',
    'a tattoo of the {}.',
]

In [98]:
mickey_prompts = [s.format('Snoopy') for s in imagenet_templates]

In [99]:
for s in mickey_prompts:
    print(repr(s))

'a bad photo of a Snoopy.'
'a photo of many Snoopy.'
'a sculpture of a Snoopy.'
'a photo of the hard to see Snoopy.'
'a low resolution photo of the Snoopy.'
'a rendering of a Snoopy.'
'graffiti of a Snoopy.'
'a bad photo of the Snoopy.'
'a cropped photo of the Snoopy.'
'a tattoo of a Snoopy.'
'the embroidered Snoopy.'
'a photo of a hard to see Snoopy.'
'a bright photo of a Snoopy.'
'a photo of a clean Snoopy.'
'a photo of a dirty Snoopy.'
'a dark photo of the Snoopy.'
'a drawing of a Snoopy.'
'a photo of my Snoopy.'
'the plastic Snoopy.'
'a photo of the cool Snoopy.'
'a close-up photo of a Snoopy.'
'a black and white photo of the Snoopy.'
'a painting of the Snoopy.'
'a painting of a Snoopy.'
'a pixelated photo of the Snoopy.'
'a sculpture of the Snoopy.'
'a bright photo of the Snoopy.'
'a cropped photo of a Snoopy.'
'a plastic Snoopy.'
'a photo of the dirty Snoopy.'
'a jpeg corrupted photo of a Snoopy.'
'a blurry photo of the Snoopy.'
'a photo of the Snoopy.'
'a good photo of the Snoop